# 01 · Data quality

What this notebook checks, on every source as published (before any correction):

1. **Coverage**: first / last month of each series, publication lag at the download date.
2. **Missing values**: leading, interior and trailing gaps.
3. **Wrong values**: rule checks (dates, signs) and a robust scan of monthly changes; each flag gets a verdict.
4. **Internal consistency**: derived columns (`D01`, `ED01`, ...) against their inputs.
5. **Cross-source checks**: IMF vs Westmetall LME prices; Eurostat steel PPI vs MEPS / CRU CR coil.

The transformations themselves live in `src/preprocess.py`. This notebook only provides the evidence. The one decision
it feeds back is the `CORRECTIONS` list in `preprocess.py`. To run it: `uv sync` (installs the kernel), then open the
notebook in VS Code and pick the `.venv` Python kernel.

> Outputs show challenge data. The pre-commit hook (nbstripout) removes them from every commit, so run the notebook again after committing.

In [ ]:
import logging
import sys
sys.path.insert(0, "../src")
logging.getLogger("matplotlib.font_manager").setLevel(logging.ERROR)  # eda's preferred fonts may be missing

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import preprocess as P
from eda import AXIS, GRID, INK2, MUTED, SLOT  # same chart style as the EDA figures

%matplotlib inline

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 60)

log = P.Log()
raw, meta = P.load_sources(log)              # as published, month-start index, nothing cut or corrected
panel, catalog, steps = P.build(save=False)  # after preprocessing
cut = raw.loc[:P.CUTOFF]                      # raw values the model is allowed to see
print(f"raw: {raw.shape[1]} series, {raw.index.min():%Y-%m} to {raw.index.max():%Y-%m}; cutoff {P.CUTOFF:%Y-%m}")

## 1. Coverage

One row per series of the final panel. `last_in_file` is the last month in the downloaded file; `lag_m` is the number
of months between it and the download month (Nov-2026), minus one, i.e. a rough publication lag.

In [ ]:
catalog[["id", "name", "source", "unit", "first", "last", "n_obs", "last_in_file", "lag_m"]]

In [ ]:
# Availability map: one row per series, dark = value present. Vertical line = cutoff.
avail = raw.notna().T
fig, ax = plt.subplots(figsize=(11, 12))
ax.imshow(avail.values, aspect="auto", cmap=plt.matplotlib.colors.ListedColormap(["#f0efec", SLOT[0]]),
          interpolation="none")
ax.set_yticks(range(len(avail)), avail.index, fontsize=7)
years = [i for i, m in enumerate(raw.index) if m.month == 1]
ax.set_xticks(years, [raw.index[i].year for i in years], fontsize=8)
ax.axvline(raw.index.get_loc(P.CUTOFF) + 0.5, color=INK2, lw=1.5)
ax.grid(False)
ax.set_title("Which months each series covers (raw files)")
plt.show()

**Reading:**
- The public files (`data.zip`) start in Nov-2016. CRU / MEPS go back to 2008–2012, Westmetall to Jan-2016, and the
  epoxy file starts in Jan-2018. The panel keeps each series' full history, so the common window is **Nov-2016 → Sep-2026**.
- `L01`–`L08` (licensed) are empty on purpose and are dropped.
- Months after Sep-2026 exist in several files (FX, power, CRU, MEPS, epoxy, Westmetall Oct-2026 + a provisional
  Nov-2026). They are dropped **before** any fill, so no later value can leak into an interpolation.

## 2. Missing values

In [ ]:
def gap_profile(s, start="2016-11-01"):
    """Counts of missing months inside [start, cutoff], split by position."""
    s = s.loc[start:P.CUTOFF]
    if s.notna().sum() == 0:
        return {"leading": len(s), "interior": 0, "trailing": 0, "interior_gaps": ""}
    first, last = s.first_valid_index(), s.last_valid_index()
    gaps = P.interior_gaps(s)
    return {"leading": int(s.loc[:first].isna().sum()), "interior": sum(n for *_, n in gaps),
            "trailing": int(s.loc[last:].isna().sum()),
            "interior_gaps": ", ".join(f"{a:%Y-%m}..{b:%Y-%m}" for a, b, _ in gaps)}

gaps = pd.DataFrame({c: gap_profile(raw[c]) for c in raw.columns}).T
gaps[(gaps[["leading", "interior", "trailing"]] != 0).any(axis=1)]

Within the common window Nov-2016 → Sep-2026:

- **No interior gaps in any source.** The only interior NaN in the panel are the two values set to NaN by the
  corrections in §3; they are filled by log-linear interpolation (`MAX_GAP = 2` months; longer gaps would stay NaN).
- **Leading:** `L01`–`L08` (empty) and the epoxy file (starts Jan-2018). Not filled: no back-casting.
- **Trailing (ragged edge):** `E20`, `E21`, `ED06` end Jul-2026 and stainless 304 ends Aug-2026. They stay NaN in the panel.
  The model step decides how to handle them (e.g. `P.as_of`, or carrying the last value forward).
- The CRU weekly columns contain `-` for missing weeks; we only read CRU's own monthly `AVERAGE` column, so this
  does not reach the panel.

## 3. Wrong values

### 3.1 Rule checks

In [ ]:
print(steps[steps.step.isin(["calendar", "drop"])].to_string(index=False))
nonpos = cut.le(0).sum()
print("\nnon-positive values:", nonpos[nonpos > 0].to_dict())
cut.loc[cut["ss304_base_de"] <= 0, ["ss304_base_de", "ss304_alloy_de"]].assign(total=lambda d: d.sum(axis=1))

- The epoxy file dates move from the 1st to the 2nd–5th of the month from Dec-2023 on; the stainless sheet uses the 15th.
  All are mapped to the first of their month, and there is never more than one row per month.
- **`ss304_base_de` is negative in Jun–Jul 2023. This is not an error:** CRU's base price is the transaction price
  minus the alloy surcharge, and in mid-2023 the surcharge was above the transaction price. The total (`ss304_de`)
  stays positive. Use the total for modelling and never take the log of the base.

### 3.2 Robust scan of monthly changes

We flag a month when its log change is more than 8 robust standard deviations away from the median change of the
25 months around it (median absolute deviation, floored at 1% so flat steel series do not divide by zero). The rolling
window keeps calm and turbulent periods apart. *Reverts* = the next month moves back, the usual pattern of a typo.
Stainless is scanned on its total: the base price alone is near zero in 2023, so its percentage changes mean nothing.

In [ ]:
X = cut.drop(columns=["ss304_base_de", "ss304_alloy_de"] + [c for c in cut if c.startswith("L")])
X["ss304_de"] = cut["ss304_base_de"] + cut["ss304_alloy_de"]
R = np.log(X.where(X > 0)).diff()
med = R.rolling(25, center=True, min_periods=12).median()
mad = (R - med).abs().rolling(25, center=True, min_periods=12).median().clip(lower=0.01)
z = (R - med) / (1.4826 * mad)
flags = z.abs().stack().rename_axis(["month", "series"]).rename("z").reset_index()
flags = flags[flags.z > 8].sort_values("z", ascending=False)
flags["change"] = [np.expm1(R.at[m, s]) for m, s in zip(flags.month, flags.series)]
flags["reverts"] = [bool(R.at[m, s] * R[s].shift(-1).at[m] < -0.5 * R.at[m, s] ** 2) for m, s in zip(flags.month, flags.series)]

VERDICT = {  # (series, month) -> verdict, after looking at the values around the flag
    ("ech", "2026-05"): "ERROR: 257 vs 2 479 / 2 607 around it, ~10x too low -> corrected",
    ("ech", "2026-06"): "rebound after the May-2026 typo",
    ("epoxy", "2026-05"): "ERROR: built from ECH, same typo -> corrected",
    ("epoxy", "2026-06"): "rebound after the May-2026 typo",
    ("E12", "2021-01"): "real: Japan cold snap, power price spike", ("E12", "2021-02"): "real: end of the spike",
    ("ED05", "2021-01"): "real: E12 in EUR", ("ED05", "2021-02"): "real: E12 in EUR",
    ("WM_AL_STOCK", "2024-05"): "real: large LME aluminium deliveries (stocks, not a price)",
    ("ech", "2021-01"): "kept: -26% and stays at the new level (no rebound), epoxy keeps rising",
    ("acetone", "2026-04"): "real: 2026 oil shock passing into chemicals (benzene +37% too)",
    ("E03", "2026-03"): "real: 2026 oil shock, Brent and WTI +43% / +42% the same month",
    ("S11", "2026-04"): "real: EU plastics PPI follows the oil shock one month later",
}
flags["verdict"] = [VERDICT.get((s, f"{m:%Y-%m}"), "") for m, s in zip(flags.month, flags.series)]
flags.assign(month=flags.month.dt.strftime("%Y-%m"), z=flags.z.round(1), change=flags.change.map("{:+.0%}".format))

In [ ]:
# The one confirmed error: ECH and epoxy around May-2026, raw vs preprocessed
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharex=True)
for ax, s in zip(axes, ["ech", "epoxy"]):
    r, p = raw[s].loc["2025-06":P.CUTOFF], panel[s].loc["2025-06":]
    ax.plot(r.index, r, color=MUTED, lw=1.5, ls="--", label="as published")
    ax.plot(p.index, p, color=SLOT[0], lw=2, label="after preprocessing")
    ax.set_title(f"{s} (EUR/t)")
    ax.legend(loc="lower left")
    ax.xaxis.set_major_locator(plt.matplotlib.dates.MonthLocator(bymonth=[1, 4, 7, 10]))
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter("%b-%y"))
plt.show()
steps[steps.step.isin(["correct", "fill"])]

**Verdict:** only one error. ECH (and therefore epoxy) in May-2026 is about 10× too low, a decimal-shift typo. It is
in `CORRECTIONS` (set to NaN, then interpolated). Every other flag lines up with a known market event, or shows up in
several related series in the same month, so it is kept.

With a lower threshold (6), more real events appear: the 2022 nickel squeeze, the Apr-2020 COVID oil crash, Russian
crude imports to the EU falling to almost zero in Feb-2026. These are the regimes for the EDA, not data errors.

## 4. Internal consistency of derived columns

The data provider derived some columns in Excel. We recompute them from their inputs.

In [ ]:
checks = {
    "D01 = S01 / S06": cut.D01 / (cut.S01 / cut.S06),
    "D02 = S02 / S06": cut.D02 / (cut.S02 / cut.S06),
    "ED01 = E01 / E23": cut.ED01 / (cut.E01 / cut.E23),
    "ED02 = E05 x 3.412 / E23": cut.ED02 / (cut.E05 * 3.412 / cut.E23),
    "ED03 = E07 x 3.412 / E23": cut.ED03 / (cut.E07 * 3.412 / cut.E23),
    "ED04 = E06 x 3.412 / E23": cut.ED04 / (cut.E06 * 3.412 / cut.E23),
    "ED05 = E12 x 1000 / E24": cut.ED05 / (cut.E12 * 1000 / cut.E24),
    "ED06 = E20 / E21": cut.ED06 / (cut.E20 / cut.E21),
    "E23 = S06": cut.E23 / cut.S06,
    "WM_FX = S06 (ECB, two publishers)": cut.WM_FX / cut.S06,
}
pd.DataFrame({k: {"max |ratio - 1|": (v - 1).abs().max(), "n": v.count()} for k, v in checks.items()}).T

All derived columns match their formula to rounding (< 0.01%). The one exception is `ED06` (up to 1.2%), which is
stored with 4 decimals while the share fell to about 1% in 2026, so the gap is rounding too. E23 is an exact copy of S06, so it is dropped. Westmetall's
ECB rate matches S06 to < 0.1%, which is why it is used to convert Westmetall's 2016 months, before S06 starts.

## 5. Cross-source checks

### 5.1 LME copper and aluminium: IMF (via FRED) vs Westmetall

In [ ]:
pairs = {"Copper": ("S01", "WM_CU_CASH"), "Aluminium": ("S02", "WM_AL_CASH")}
fig, ax = plt.subplots(figsize=(10, 3.4))
rows = {}
for (name, (a, b)), c in zip(pairs.items(), [SLOT[2], SLOT[1]]):
    gap = (panel[a] / panel[b] - 1).dropna() * 100
    ax.plot(gap.index, gap, color=c, lw=2, label=f"{name}: IMF / Westmetall - 1")
    ra, rb = np.log(panel[a]).diff(), np.log(panel[b]).diff()
    rows[name] = {"mean gap %": gap.mean(), "max |gap| %": gap.abs().max(), "worst month": f"{gap.abs().idxmax():%Y-%m}",
                  "corr of monthly changes": ra.corr(rb)}
ax.axhline(0, color=AXIS, lw=1)
ax.set_ylabel("%")
ax.set_title("Same LME price from two publishers: level gap (%)")
ax.legend(loc="upper left")
plt.show()
pd.DataFrame(rows).T

The IMF series is the LME cash monthly average: the levels agree to within about 1% (the largest gaps are in the
Mar-2022 nickel-squeeze market and Jan-2026), and monthly changes correlate at > 0.99. The two sources can be used
interchangeably. Westmetall adds what the IMF series does not have: the **3-month forward price** and **LME stocks**.

### 5.2 Carbon steel: Eurostat PPI (S09) vs MEPS and CRU CR coil Germany

In [ ]:
steel = panel[["S09", "meps_cr_de", "cru_cr_de", "S10", "cru_hdg_de"]].loc["2016-11":]
reb = 100 * steel / steel.iloc[0]
fig, ax = plt.subplots(figsize=(10, 3.8))
for s, c, lab in [("S09", SLOT[0], "EU steel PPI (S09)"), ("meps_cr_de", SLOT[3], "MEPS CR coil DE"),
                  ("cru_cr_de", SLOT[1], "CRU CR coil DE")]:
    ax.plot(reb.index, reb[s], color=c, lw=2, label=lab)
ax.axhline(100, color=AXIS, lw=1)
ax.set_ylabel("index, Nov-2016 = 100")
ax.set_title("Carbon steel: EU PPI vs CR coil market prices (Nov-2016 = 100)")
ax.legend(loc="upper left")
plt.show()

L = np.log(steel)
def lead_corr(a, b, lags=range(-3, 4)):
    """corr(change of a in month t, change of b in month t-k): a peak at k > 0 means b leads a by k months."""
    return {k: L[a].diff().corr(L[b].diff().shift(k)) for k in lags}

tab = pd.DataFrame({f"{a} vs {b}": lead_corr(a, b) for a, b in
                    [("S09", "cru_cr_de"), ("S09", "meps_cr_de"), ("meps_cr_de", "cru_cr_de"), ("S10", "cru_hdg_de")]}).T
tab.columns = [f"k={k}" for k in tab.columns]
tab["swing ratio (std 12m change, a / b)"] = [
    (L[a] - L[a].shift(12)).std() / (L[b] - L[b].shift(12)).std() for a, b in
    [("S09", "cru_cr_de"), ("S09", "meps_cr_de"), ("meps_cr_de", "cru_cr_de"), ("S10", "cru_hdg_de")]]
display(tab.round(2))
reb.loc[pd.to_datetime(["2021-01", "2022-04", "2022-05", "2023-06", "2024-12", "2026-09"])].round(0)

- **Same trend, different timing:** monthly changes of the PPI correlate best with the CRU / MEPS change of the
  **previous** month (k = 1, r ≈ 0.7, against ≈ 0.56 in the same month). The market price leads the PPI by about one
  month, which is useful for the 3-month forecast of S09 / S10.
- **Smaller and slower swings in the PPI** (swing ratio ≈ 0.63): from Nov-2016, CR coil peaks at about 250 in Apr-2022
  and the PPI at about 220 a month later. On the way down the PPI is sticky: in Jun-2023 it is still at ~170 while CR
  coil is back at ~135. The PPI is a broad basket (all steel products, ferro-alloys, contract prices). A BOM cost index
  built on S09 therefore smooths and delays the real CR-coil price risk; CRU / MEPS are the better EUR/t level.
- MEPS and CRU agree closely (r ≈ 0.94 in the same month), so either can be the EUR/t level for carbon steel.

## 6. Decisions passed to `preprocess.py`

| Finding | Action | Where |
|---|---|---|
| Values after Sep-2026 in several files | Dropped before any fill | step 2 `cutoff` |
| Westmetall Nov-2026 is month-to-date | Dropped | `load_sources` |
| `L01`–`L08` empty, `E23` = `S06` | Dropped | step 3 `drop` |
| ECH / epoxy May-2026 typo | Set to NaN, then log-linear interpolation | `CORRECTIONS`, step 5 |
| No other interior gaps | Fill rule kept for robustness (≤ 2 months) | step 5 |
| Ragged edge (E20/E21/ED06 Jul, stainless Aug) | Left NaN; lag stored in `series_catalog.csv` (`lag_m`) | `as_of()` helper |
| Negative stainless base price 2023 | Kept (real); use `ss304_de` total | step 6 |
| USD prices without EUR twin | Converted with ECB monthly mean (`*_EUR`) | step 7 |
| IMF and Westmetall LME agree | Both kept; Westmetall adds 3-month price and stocks | — |